# Boosted Balanced Weight Tuning

`submission_boosted_balanced.csv` is the current best public result. This notebook tunes near the balanced class weights and writes multiple nearby submissions for leaderboard testing.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42
ID_COL = "id"
TARGET_COL = "health_condition"

## Load Data

In [2]:
train_df = pd.read_csv("data/train_split_features_numeric.csv")
val_df = pd.read_csv("data/val_split_features_numeric.csv")
test_df = pd.read_csv("data/test_features_numeric.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]
X_train = train_df[feature_cols].astype("float32")
X_val = val_df[feature_cols].astype("float32")
X_test = test_df[feature_cols].astype("float32")
y_train_raw = train_df[TARGET_COL]
y_val_raw = val_df[TARGET_COL]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
class_names = label_encoder.classes_
class_id_by_name = {name: int(label_encoder.transform([name])[0]) for name in class_names}

balanced_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=y_train,
)
balanced_by_name = {name: float(balanced_array[class_id_by_name[name]]) for name in class_names}

print("class mapping:", class_id_by_name)
print("balanced weights from train split:", balanced_by_name)
print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

class mapping: {'at-risk': 0, 'fit': 1, 'unhealthy': 2}
balanced weights from train split: {'at-risk': 0.38819475061298164, 'fit': 5.779264284069258, 'unhealthy': 3.985000397005854}
train: (552070, 73) val: (138018, 73) test: (295753, 72)


## Class-Weight Variants Near Balanced

In [3]:
variant_configs = [
    {"name": "balanced_exact", "weights_by_name": balanced_by_name},
    {"name": "balanced_less_minority", "weights_by_name": {"at-risk": 0.42, "fit": 5.30, "unhealthy": 3.60}},
    {"name": "balanced_fit_up", "weights_by_name": {"at-risk": 0.388, "fit": 6.40, "unhealthy": 3.985}},
    {"name": "balanced_unhealthy_up", "weights_by_name": {"at-risk": 0.388, "fit": 5.779, "unhealthy": 4.50}},
    {"name": "balanced_both_up", "weights_by_name": {"at-risk": 0.36, "fit": 6.30, "unhealthy": 4.40}},
    {"name": "balanced_fit_down_unhealthy_up", "weights_by_name": {"at-risk": 0.40, "fit": 5.10, "unhealthy": 4.70}},
]

pd.DataFrame(variant_configs)

,name,weights_by_name
0,balanced_exact,"{'at-risk': 0.38819475061298164, 'fit': 5.779264284069258, 'unhealthy': 3.985000397005854}"
1,balanced_less_minority,"{'at-risk': 0.42, 'fit': 5.3, 'unhealthy': 3.6}"
2,balanced_fit_up,"{'at-risk': 0.388, 'fit': 6.4, 'unhealthy': 3.985}"
3,balanced_unhealthy_up,"{'at-risk': 0.388, 'fit': 5.779, 'unhealthy': 4.5}"
4,balanced_both_up,"{'at-risk': 0.36, 'fit': 6.3, 'unhealthy': 4.4}"
5,balanced_fit_down_unhealthy_up,"{'at-risk': 0.4, 'fit': 5.1, 'unhealthy': 4.7}"


## Validation Training

In [4]:
def class_weight_from_names(weights_by_name):
    return {class_id_by_name[name]: weight for name, weight in weights_by_name.items()}


def make_model(class_weight, max_iter=500, early_stopping=True):
    return HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.06,
        max_iter=max_iter,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        early_stopping=early_stopping,
        validation_fraction=0.15 if early_stopping else None,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
        class_weight=class_weight,
        verbose=0,
    )

validation_rows = []
validation_models = {}

for config in variant_configs:
    class_weight = class_weight_from_names(config["weights_by_name"])
    print("Training", config["name"], class_weight)
    model = make_model(class_weight=class_weight)
    model.fit(X_train, y_train)

    val_pred = label_encoder.inverse_transform(model.predict(X_val))
    val_distribution = pd.Series(val_pred).value_counts(normalize=True).mul(100).to_dict()

    validation_rows.append({
        "variant": config["name"],
        "accuracy": accuracy_score(y_val_raw, val_pred),
        "macro_f1": f1_score(y_val_raw, val_pred, average="macro"),
        "weighted_f1": f1_score(y_val_raw, val_pred, average="weighted"),
        "n_iter": model.n_iter_,
        "val_at_risk_pct": val_distribution.get("at-risk", 0),
        "val_fit_pct": val_distribution.get("fit", 0),
        "val_unhealthy_pct": val_distribution.get("unhealthy", 0),
    })
    validation_models[config["name"]] = model

validation_report = pd.DataFrame(validation_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
validation_report

Training balanced_exact {0: 0.38819475061298164, 1: 5.779264284069258, 2: 3.985000397005854}


Training balanced_less_minority {0: 0.42, 1: 5.3, 2: 3.6}


Training balanced_fit_up {0: 0.388, 1: 6.4, 2: 3.985}


Training balanced_unhealthy_up {0: 0.388, 1: 5.779, 2: 4.5}


Training balanced_both_up {0: 0.36, 1: 6.3, 2: 4.4}


Training balanced_fit_down_unhealthy_up {0: 0.4, 1: 5.1, 2: 4.7}


,variant,accuracy,macro_f1,weighted_f1,n_iter,val_at_risk_pct,val_fit_pct,val_unhealthy_pct
0,balanced_less_minority,0.902940,0.795538,0.910924,130,78.510049,9.187207,12.302743
1,balanced_fit_down_unhealthy_up,0.889203,0.778827,0.899651,195,76.808097,9.072005,14.119897
2,balanced_exact,0.882624,0.766829,0.894358,123,76.074860,10.278369,13.646771
3,balanced_fit_up,0.876480,0.757949,0.889501,123,75.380023,10.953644,13.666333
4,balanced_unhealthy_up,0.875516,0.758313,0.888553,123,75.224246,10.323291,14.452463
5,balanced_both_up,0.864742,0.743890,0.879964,131,73.959918,11.195641,14.844441


## Final Full-Data Training And Submissions

In [5]:
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = full_df[feature_cols].astype("float32")
y_full = label_encoder.transform(full_df[TARGET_COL])

submission_rows = []

for config in variant_configs:
    variant = config["name"]
    class_weight = class_weight_from_names(config["weights_by_name"])
    final_max_iter = int(validation_models[variant].n_iter_)

    print("Final training", variant, "max_iter", final_max_iter, class_weight)
    final_model = make_model(class_weight=class_weight, max_iter=final_max_iter, early_stopping=False)
    final_model.fit(X_full, y_full)

    test_pred = label_encoder.inverse_transform(final_model.predict(X_test))
    submission = sample_submission.copy()
    submission[ID_COL] = test_df[ID_COL].values
    submission[TARGET_COL] = test_pred

    output_path = f"data/submission_tuned_{variant}.csv"
    submission.to_csv(output_path, index=False)

    test_distribution = submission[TARGET_COL].value_counts(normalize=True).mul(100).to_dict()
    submission_rows.append({
        "variant": variant,
        "path": output_path,
        "test_at_risk_pct": test_distribution.get("at-risk", 0),
        "test_fit_pct": test_distribution.get("fit", 0),
        "test_unhealthy_pct": test_distribution.get("unhealthy", 0),
    })

submission_report = pd.DataFrame(submission_rows)
submission_report

Final training balanced_exact max_iter 123 {0: 0.38819475061298164, 1: 5.779264284069258, 2: 3.985000397005854}


Final training balanced_less_minority max_iter 130 {0: 0.42, 1: 5.3, 2: 3.6}


Final training balanced_fit_up max_iter 123 {0: 0.388, 1: 6.4, 2: 3.985}


Final training balanced_unhealthy_up max_iter 123 {0: 0.388, 1: 5.779, 2: 4.5}


Final training balanced_both_up max_iter 131 {0: 0.36, 1: 6.3, 2: 4.4}


Final training balanced_fit_down_unhealthy_up max_iter 195 {0: 0.4, 1: 5.1, 2: 4.7}


,variant,path,test_at_risk_pct,test_fit_pct,test_unhealthy_pct
0,balanced_exact,data/submission_tuned_balanced_exact.csv,75.610391,10.526351,13.863258
1,balanced_less_minority,data/submission_tuned_balanced_less_minority.csv,78.042826,9.421713,12.535460
2,balanced_fit_up,data/submission_tuned_balanced_fit_up.csv,74.993322,11.169117,13.837560
3,balanced_unhealthy_up,data/submission_tuned_balanced_unhealthy_up.csv,74.887829,10.517222,14.594949
4,balanced_both_up,data/submission_tuned_balanced_both_up.csv,73.556312,11.519410,14.924278
5,balanced_fit_down_unhealthy_up,data/submission_tuned_balanced_fit_down_unhealthy_up.csv,76.354931,9.326702,14.318367


## Combined Report

In [6]:
combined_report = validation_report.merge(submission_report, on="variant", how="left")
combined_report

,variant,accuracy,macro_f1,weighted_f1,n_iter,val_at_risk_pct,val_fit_pct,val_unhealthy_pct,path,test_at_risk_pct,test_fit_pct,test_unhealthy_pct
0,balanced_less_minority,0.902940,0.795538,0.910924,130,78.510049,9.187207,12.302743,data/submission_tuned_balanced_less_minority.csv,78.042826,9.421713,12.535460
1,balanced_fit_down_unhealthy_up,0.889203,0.778827,0.899651,195,76.808097,9.072005,14.119897,data/submission_tuned_balanced_fit_down_unhealthy_up.csv,76.354931,9.326702,14.318367
2,balanced_exact,0.882624,0.766829,0.894358,123,76.074860,10.278369,13.646771,data/submission_tuned_balanced_exact.csv,75.610391,10.526351,13.863258
3,balanced_fit_up,0.876480,0.757949,0.889501,123,75.380023,10.953644,13.666333,data/submission_tuned_balanced_fit_up.csv,74.993322,11.169117,13.837560
4,balanced_unhealthy_up,0.875516,0.758313,0.888553,123,75.224246,10.323291,14.452463,data/submission_tuned_balanced_unhealthy_up.csv,74.887829,10.517222,14.594949
5,balanced_both_up,0.864742,0.743890,0.879964,131,73.959918,11.195641,14.844441,data/submission_tuned_balanced_both_up.csv,73.556312,11.519410,14.924278
